# 🎙️ Fine-tuning Whisper — reconnaissance du créole guadeloupéen

Notebook **clé en main** : entraîne Whisper à transcrire l'audio créole, puis exporte le modèle pour l'app **Texty**.

### Avant de lancer
1. **Runtime ▸ Modifier le type d'exécution ▸ GPU** (T4 suffit).
2. Prépare tes données dans Google Drive : un dossier contenant tes fichiers audio **et** un `manifest.csv` avec 2 colonnes :
   `audio_path,text` (audio ~16 kHz mono ; `audio_path` relatif au dossier, `text` = transcription créole validée).
3. Puis **Exécution ▸ Tout exécuter**. À la fin tu télécharges `whisper-creole-ct2.zip`.

> ⚠️ La qualité dépend surtout de la **quantité de données** (viser plusieurs heures d'audio transcrit).

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi -L || print("⚠️ Aucun GPU détecté — Runtime ▸ Modifier le type d'exécution ▸ GPU")

## 2. Installer les dépendances

In [ ]:
!pip -q install "transformers>=4.40" "datasets>=2.18" evaluate jiwer librosa soundfile accelerate "ctranslate2>=4.0" sentencepiece

## 3. Monter Google Drive et régler les paramètres

👉 **Édite les valeurs ci-dessous** selon ton dossier de données.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ======== À ÉDITER ========
DATA_DIR   = "/content/drive/MyDrive/creole_asr"   # dossier : audios + manifest.csv
MANIFEST   = f"{DATA_DIR}/manifest.csv"            # colonnes : audio_path,text
BASE_MODEL = "openai/whisper-small"                # tiny | base | small | medium
LANGUAGE   = "French"                              # langue proxy pour le créole
OUTPUT_DIR = "/content/whisper-creole"
EPOCHS     = 10
BATCH_SIZE = 8
LR         = 1e-5
# ==========================

## 4. Charger le manifeste et préparer train / dev

In [ ]:
import os, csv, random
from datasets import Dataset, Audio

def load_manifest(path, data_dir):
    rows = []
    with open(path, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            a = (r.get("audio_path") or "").strip()
            t = (r.get("text") or "").strip()
            if not a or not t:
                continue
            if not os.path.isabs(a) and not os.path.exists(a):
                a = os.path.join(data_dir, a)
            if not os.path.exists(a):
                print("⚠️ audio introuvable:", a); continue
            rows.append({"audio": a, "text": t})
    return rows

rows = load_manifest(MANIFEST, DATA_DIR)
assert rows, "Aucune donnée valide : vérifie DATA_DIR et manifest.csv"
random.Random(42).shuffle(rows)
n_dev = max(1, int(len(rows) * 0.1))
dev_rows, train_rows = rows[:n_dev], rows[n_dev:]

def to_ds(rs):
    ds = Dataset.from_dict({"audio": [r["audio"] for r in rs], "text": [r["text"] for r in rs]})
    return ds.cast_column("audio", Audio(sampling_rate=16000))

train_ds, dev_ds = to_ds(train_rows), to_ds(dev_rows)
print(f"train = {len(train_ds)}   dev = {len(dev_ds)}")

## 5. Charger Whisper (processeur + modèle)

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained(BASE_MODEL, language=LANGUAGE, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
model.generation_config.language = LANGUAGE.lower()
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

## 6. Prétraitement + collateur + métrique WER

In [ ]:
import torch, evaluate
from dataclasses import dataclass
from typing import Any

def prepare(b):
    a = b["audio"]
    b["input_features"] = processor.feature_extractor(a["array"], sampling_rate=16000).input_features[0]
    b["labels"] = processor.tokenizer(b["text"]).input_ids
    return b

train_ds = train_ds.map(prepare, remove_columns=train_ds.column_names)
dev_ds   = dev_ds.map(prepare, remove_columns=dev_ds.column_names)

@dataclass
class Collator:
    processor: Any
    def __call__(self, features):
        inp = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(inp, return_tensors="pt")
        lab = [{"input_ids": f["labels"]} for f in features]
        lb = self.processor.tokenizer.pad(lab, return_tensors="pt")
        labels = lb["input_ids"].masked_fill(lb.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

collator = Collator(processor)
wer = evaluate.load("wer")

def compute_metrics(pred):
    pid, lid = pred.predictions, pred.label_ids
    lid[lid == -100] = processor.tokenizer.pad_token_id
    ps = processor.tokenizer.batch_decode(pid, skip_special_tokens=True)
    ls = processor.tokenizer.batch_decode(lid, skip_special_tokens=True)
    return {"wer": 100 * wer.compute(predictions=ps, references=ls)}

## 7. Entraîner

La métrique **WER** (Word Error Rate) s'affiche à chaque époque — **plus bas = mieux**.

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    generation_max_length=225,
    logging_steps=20,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=dev_ds,
    data_collator=collator, compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("✅ Modèle affiné ->", OUTPUT_DIR)

## 8. Test rapide sur un extrait

In [ ]:
import librosa
s = dev_rows[0]
audio, _ = librosa.load(s["audio"], sr=16000, mono=True)
inp = processor(audio, sampling_rate=16000, return_tensors="pt").input_features.to(model.device)
ids = model.generate(inp)
print("Référence :", s["text"])
print("Modèle    :", processor.batch_decode(ids, skip_special_tokens=True)[0])

## 9. Convertir au format faster-whisper (utilisé par l'app)

In [ ]:
!ct2-transformers-converter --model {OUTPUT_DIR} --output_dir {OUTPUT_DIR}-ct2 --quantization int8 --force
print("✅ Modèle faster-whisper ->", OUTPUT_DIR + "-ct2")

## 10. Sauvegarder (Drive) + télécharger le modèle

In [ ]:
import shutil

# a) Copie dans ton Drive
dst = "/content/drive/MyDrive/whisper-creole-ct2"
shutil.rmtree(dst, ignore_errors=True)
shutil.copytree(OUTPUT_DIR + "-ct2", dst)
print("📁 Copié dans Drive :", dst)

# b) Archive téléchargeable
shutil.make_archive("/content/whisper-creole-ct2", "zip", OUTPUT_DIR + "-ct2")
from google.colab import files
files.download("/content/whisper-creole-ct2.zip")

## 11. Brancher le modèle dans l'app Texty

1. Dézippe `whisper-creole-ct2.zip` et place le dossier dans `models/` du projet.
2. Backend : définis la variable d'environnement
   `CREOLE_MODEL_PATH=models/whisper-creole-ct2` (en local)
   ou, sur **Hugging Face Space** : Settings ▸ Variables → `CREOLE_MODEL_PATH=/app/models/whisper-creole-ct2`
   (embarque le dossier dans l'image Docker, ou télécharge-le au démarrage).
3. Le backend chargera **ton modèle créole affiné à la place** du Whisper standard. 🎉